# Semantic Clothing Recommender — Preprocessing & Indexing

This notebook prepares the Myntra Fashion Dataset for semantic search:

1. **Load & inspect** the raw CSV
2. **Clean** missing / malformed values
3. **Build a text representation** for each product
4. **Generate embeddings** using SentenceTransformers
5. **Build a FAISS index** and save it to disk

**Prerequisites**
- Download the dataset from Kaggle:  
  https://www.kaggle.com/datasets/hiteshsuthar101/myntra-fashion-product-dataset  
  Place `Fashion Dataset.csv` inside the `data/` folder.
- Install dependencies:  `pip install -r requirements.txt`

Run all cells from top to bottom — the index is saved to `index/` and is
picked up automatically by `app.py`.

## 1 · Imports & configuration

In [ ]:
import re
import sys
import pathlib

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Make sure the project root is on the path so config and src are importable
PROJECT_ROOT = pathlib.Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from src.encoder import TextEncoder
from src.vector_store import VectorStore

print("Project root :", PROJECT_ROOT)
print("Dataset path :", config.DATASET_CSV)
print("FAISS index  :", config.FAISS_INDEX_PATH)
print("Embedding dim:", config.EMBEDDING_DIM)

## 2 · Load the raw dataset

In [ ]:
df = pd.read_csv(config.DATASET_CSV, on_bad_lines="skip")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

In [ ]:
print("Column dtypes:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

## 3 · Data cleaning

In [ ]:
# Drop rows with no name or no description — they provide no semantic signal
df = df.dropna(subset=["name", "description"]).reset_index(drop=True)

# Strip HTML tags from descriptions (some entries contain <br> etc.)
HTML_TAG = re.compile(r"<[^>]+>")

def clean_text(text: str) -> str:
    text = HTML_TAG.sub(" ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["description"] = df["description"].apply(clean_text)
df["name"] = df["name"].apply(clean_text)

# Normalise numeric columns
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["avg_rating"] = pd.to_numeric(df["avg_rating"], errors="coerce")
df["ratingCount"] = pd.to_numeric(df["ratingCount"], errors="coerce")

# Fill remaining NaNs with empty strings for text fields
for col in ["colour", "brand", "img", "p_attributes"]:
    df[col] = df[col].fillna("")

print(f"Clean dataset: {len(df):,} rows")
df[["name", "brand", "colour", "price", "avg_rating"]].describe()

## 4 · Build product text representations

Each product is converted to a single descriptive sentence that the
embedding model can encode into a dense vector.

In [ ]:
encoder = TextEncoder(model_name=config.TEXT_EMBEDDING_MODEL)

tqdm.pandas(desc="Building product texts")
df["product_text"] = df.progress_apply(
    lambda row: encoder.build_product_text(row), axis=1
)

print("Sample product texts:")
for i, text in enumerate(df["product_text"].head(3)):
    print(f"\n[{i}] {text[:300]}")

## 5 · Generate embeddings

This cell can take a few minutes on CPU for large datasets.
On a GPU it completes in seconds.

In [ ]:
texts = df["product_text"].tolist()
print(f"Encoding {len(texts):,} product texts …")

embeddings = encoder.encode(texts, batch_size=128)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")
print(f"Sample norms (should be ≈1.0): {np.linalg.norm(embeddings[:5], axis=1)}")

## 6 · Build FAISS index & save

In [ ]:
# Columns to keep in the metadata store (used for display in the app)
META_COLS = ["p_id", "name", "price", "colour", "brand",
             "img", "ratingCount", "avg_rating", "description"]

metadata = df[META_COLS].to_dict(orient="records")

store = VectorStore(dim=config.EMBEDDING_DIM)
store.add(embeddings, metadata)

store.save(
    index_path=config.FAISS_INDEX_PATH,
    meta_path=config.METADATA_PATH,
)

print(f"\n✅ Index saved! Total vectors: {len(store):,}")

## 7 · Quick sanity check — test a sample query

In [ ]:
# Reload from disk to verify persistence
store_reloaded = VectorStore.load(
    index_path=config.FAISS_INDEX_PATH,
    meta_path=config.METADATA_PATH,
)

QUERY = "blue floral kurta for summer"
query_emb = encoder.encode(QUERY)
results = store_reloaded.search(query_emb, top_k=5)

print(f"Top 5 results for: '{QUERY}'\n")
for i, item in enumerate(results, 1):
    print(
        f"{i}. [{item['score']:.3f}] {item['name']} "
        f"| {item['brand']} | {item['colour']} | ₹{item['price']}"
    )

## Done!

The FAISS index is ready. Launch the Streamlit app with:

```bash
streamlit run app.py
```